# LeetCode 2243 — Calculate Digit Sum of a String bằng Python

Link đề: https://leetcode.com/problems/calculate-digit-sum-of-a-string/ · Độ khó: **Easy** · Chủ đề: String, Simulation

Skeleton LeetCode:

```python
class Solution:
    def digitSum(self, s: str, k: int) -> str:
        ...
```

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề dạng **mô phỏng (simulation)**: lặp một quy trình cho đến khi điều kiện dừng thỏa.
2. Cắt chuỗi thành các **nhóm liên tiếp độ dài `k`** (nhóm cuối có thể ngắn hơn) bằng `range(0, len(s), k)` và lát cắt `s[i:i+k]`.
3. Tính **tổng các chữ số** của một nhóm bằng `sum(map(int, group))` và nối kết quả bằng `"".join`.
4. Viết 5 cách: vòng `while` + lát cắt, đệ quy, tính tay từng ký tự, `iter/islice` chia khối, và biểu thức một dòng; hiểu vì sao quy trình **luôn dừng**.
5. Trace từng vòng, đối chiếu mọi cách bằng test ngẫu nhiên, và tránh lỗi như nhầm điều kiện dừng `len(s) > k`, quên nhóm cuối, hoặc cộng ký tự mà không đổi sang số.

## 2. Đề bài & ví dụ

Cho chuỗi chữ số `s` và số nguyên `k`. Một **vòng** gồm ba bước:

1. **Chia** `s` thành các nhóm liên tiếp có độ dài `k` (nhóm cuối có thể ngắn hơn nếu `len(s)` không chia hết cho `k`).
2. **Thay** mỗi nhóm bằng **tổng các chữ số** của nhóm đó (viết dưới dạng chuỗi, có thể nhiều chữ số).
3. **Ghép** các kết quả theo thứ tự ban đầu để được `s` mới.

Lặp lại các vòng **chừng nào `len(s) > k`**. Trả về `s` khi `len(s) <= k`.

| Ví dụ | s | k | Output |
|---|---|---|---|
| 1 | `"11111222223"` | `3` | `"135"` |
| 2 | `"00000000"` | `3` | `"000"` |

Ví dụ 1: vòng 1: nhóm `111,112,222,23` → tổng `3,4,6,5` → `"3465"` (dài 4 > 3). Vòng 2: nhóm `346,5` → `13,5` → `"135"` (dài 3, dừng).

Ví dụ 2: nhóm `000,000,00` → `0,0,0` → `"000"` (dài 3, dừng).

Constraints: `1 <= len(s) <= 100`, `2 <= k <= 100`, `s` chỉ gồm chữ số.

In [1]:
examples = [("11111222223", 3, "135"), ("00000000", 3, "000")]
for s, k, expected in examples:
    groups = [s[i:i + k] for i in range(0, len(s), k)]
    print(f"s={s!r}, k={k} | nhóm vòng 1: {groups} | tổng: {[sum(map(int, g)) for g in groups]} | mong đợi {expected!r}")

s='11111222223', k=3 | nhóm vòng 1: ['111', '112', '222', '23'] | tổng: [3, 4, 6, 5] | mong đợi '135'
s='00000000', k=3 | nhóm vòng 1: ['000', '000', '00'] | tổng: [0, 0, 0] | mong đợi '000'


## 3. Kiến thức nền

### 3.1. Lát cắt chuỗi `s[a:b]`

`s[a:b]` lấy các ký tự từ chỉ số `a` đến **trước** `b`. Nếu `b` vượt quá độ dài, Python **tự cắt ở cuối** chứ không báo lỗi. Đó chính là lý do `s[i:i+k]` an toàn cho **nhóm cuối ngắn hơn** `k`.

In [2]:
s = "11111222223"
print(s[0:3], s[3:6], s[9:12], s[9:100])
print(len(s[9:12]))

111 112 23 23
2


### 3.2. `range(start, stop, step)` — bước nhảy

`range(0, 11, 3)` cho `0, 3, 6, 9`: các chỉ số **bắt đầu mỗi nhóm**. Không cần biết số nhóm trước; `range` dừng khi vượt `stop`. Số nhóm là `ceil(len / k)`.

In [3]:
print(list(range(0, 11, 3)))
print(list(range(0, 8, 3)))
import math
print(math.ceil(11 / 3), (11 + 3 - 1) // 3)

[0, 3, 6, 9]
[0, 3, 6]
4 4


### 3.3. `int`, `map`, `sum`

`int("7")` đổi ký tự số thành số nguyên. `map(int, "112")` áp `int` cho từng ký tự; `sum(...)` cộng chúng. **Cẩn thận:** `sum("112")` gây lỗi vì không cộng được chuỗi.

In [4]:
print(int("7"), list(map(int, "112")), sum(map(int, "112")))
try:
    print(sum("112"))
except TypeError as e:
    print("lỗi:", e)
print(sum(int(c) for c in "222"))

7 [1, 1, 2] 4
lỗi: unsupported operand type(s) for +: 'int' and 'str'
6


### 3.4. `str(...)` và `"".join(...)`

Tổng là số nguyên nên cần `str(tổng)` để biến thành chuỗi (có thể nhiều chữ số, ví dụ `13` → `"13"`). Dùng `"".join(list_of_strings)` để nối nhiều chuỗi **hiệu quả** hơn cộng chuỗi trong vòng lặp.

In [5]:
parts = [str(3), str(4), str(13), str(5)]
print(parts, "".join(parts), len("".join(parts)))
print(str(13) + str(5))

['3', '4', '13', '5'] 34135 5
135


### 3.5. Vòng `while` và điều kiện dừng

`while điều_kiện:` lặp khi điều kiện còn đúng. Bài này lặp **khi `len(s) > k`**. Nhớ: điều kiện dừng là `len(s) <= k` — **bằng `k` cũng đã dừng**, không thực hiện thêm vòng nào.

In [6]:
n = 11
rounds = 0
while n > 3:
    n = (n + 2) // 3          # độ dài xấp xỉ sau mỗi vòng nếu tổng một chữ số
    rounds += 1
    print("vòng", rounds, "-> độ dài", n)

vòng 1 -> độ dài 4
vòng 2 -> độ dài 2


### 3.6. Đệ quy — hàm gọi chính nó

Quy trình "làm một vòng rồi làm tiếp trên kết quả" cũng là một đệ quy: `solve(s)` = `s` nếu `len(s) <= k`, ngược lại `solve(một_vòng(s))`. Cần có **trường hợp cơ sở** (điều kiện dừng) để không đệ quy vô hạn.

In [7]:
def countdown(n):
    if n == 0:
        return "xong"
    return countdown(n - 1)
print(countdown(5))

xong


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Làm tay `s = "11111222223"`, `k = 3`:

| vòng | `s` | độ dài | các nhóm | tổng các nhóm | `s` mới |
|---|---|---|---|---|---|
| 1 | `11111222223` | 11 > 3 | `111`, `112`, `222`, `23` | 3, 4, 6, 5 | `3465` |
| 2 | `3465` | 4 > 3 | `346`, `5` | 13, 5 | `135` |
| – | `135` | 3 ≤ 3 | dừng | | trả về `135` |

Mỗi vòng "nén" chuỗi: `k` chữ số thành một tổng (thường ngắn hơn). Tiếp tục nén cho đến khi chuỗi đủ ngắn (`≤ k`). Bài này không có mẹo toán học; đó là bài **mô phỏng đúng đề**, và kỹ năng cần là chia nhóm đúng và nối đúng.

In [8]:
s, k = "11111222223", 3
rounds = 0
while len(s) > k:
    rounds += 1
    groups = [s[i:i + k] for i in range(0, len(s), k)]
    sums = [sum(map(int, g)) for g in groups]
    s = "".join(map(str, sums))
    print(f"vòng {rounds}: nhóm={groups} tổng={sums} -> s={s!r}")
print("kết quả:", s)

vòng 1: nhóm=['111', '112', '222', '23'] tổng=[3, 4, 6, 5] -> s='3465'
vòng 2: nhóm=['346', '5'] tổng=[13, 5] -> s='135'
kết quả: 135


## 5. Cách 1 — Brute Force / baseline: tính tay từng ký tự, không dùng `map/sum/join`

Duyệt từng chỉ số, cộng dồn chữ số bằng `ord(c) - ord("0")`, và khi đủ `k` ký tự (hoặc hết chuỗi) thì ghi tổng vào chuỗi mới bằng phép `+`. Chưa dùng lát cắt hay hàm dựng sẵn.

In [9]:
def digit_sum_manual(s, k):
    while len(s) > k:
        new = ""
        acc = 0
        count = 0
        for ch in s:
            acc += ord(ch) - ord("0")
            count += 1
            if count == k:
                new += str(acc)
                acc = 0
                count = 0
        if count > 0:
            new += str(acc)
        s = new
    return s

print(digit_sum_manual("11111222223", 3), digit_sum_manual("00000000", 3))

135 000


In [10]:
from typing import List

class SolutionManual:
    def digitSum(self, s: str, k: int) -> str:
        while len(s) > k:
            new = ""
            acc = 0
            count = 0
            for ch in s:
                acc += ord(ch) - ord("0")
                count += 1
                if count == k:
                    new += str(acc)
                    acc = 0
                    count = 0
            if count > 0:
                new += str(acc)
            s = new
        return s

print(SolutionManual().digitSum("11111222223", 3))
print(SolutionManual().digitSum("00000000", 3))

135
000


**Độ phức tạp:** Gọi `n = len(s)`. Mỗi vòng duyệt `n` ký tự → `O(n)`; số vòng tối đa cỡ `O(log_k n)` khi chuỗi co nhanh, nhưng khi `k = 2` có thể nhiều hơn (xem mục 11). Với `n ≤ 100` rất nhỏ. **Space `O(n)`** cho chuỗi mới.

**Nút thắt:** code đúng nhưng dài, phải tự quản `acc`, `count`; nhóm cuối cần xử lý riêng (`if count > 0`). Lát cắt `s[i:i+k]` giúp bỏ hết sự rườm rà.

## 6. Ý tưởng thuật toán / bước đột phá

Insight: mỗi vòng chính là **một biểu thức** trên các nhóm do `range(0, len(s), k)` sinh ra:

- nhóm thứ `j` là `s[j*k : j*k + k]` — lát cắt tự co lại ở nhóm cuối nên không cần `if` đặc biệt;
- giá trị mới của nhóm là `str(sum(map(int, group)))`;
- `s` mới là phép nối các giá trị đó.

Quy tắc dừng: **nếu `len(s) <= k` thì trả `s`**, ngược lại làm một vòng nữa. Vì mỗi vòng hoặc làm **tổng chữ số** của cả chuỗi giảm hoặc làm **độ dài** giảm, nên chắc chắn dừng (chứng minh ở mục 10).

## 7. Các cách tiếp theo

### Cách 2 — `while` + lát cắt + `map/sum` + `join`

Cách chuẩn: gọn, dễ đọc, chính là mô tả đề.

In [11]:
from typing import List

class SolutionSlice:
    def digitSum(self, s: str, k: int) -> str:
        while len(s) > k:
            parts = []
            for i in range(0, len(s), k):
                group = s[i:i + k]
                parts.append(str(sum(map(int, group))))
            s = "".join(parts)
        return s

print(SolutionSlice().digitSum("11111222223", 3))
print(SolutionSlice().digitSum("00000000", 3))
print(SolutionSlice().digitSum("9999", 2))

135
000
99


**Độ phức tạp:** Mỗi vòng `O(n)`, `n` giảm dần; tổng thời gian `O(n · số_vòng)` (nhỏ với `n ≤ 100`). Space `O(n)`.

### Cách 3 — Đệ quy

Hàm `digitSum(s, k)` tự gọi trên chuỗi đã gộp. Trường hợp cơ sở: `len(s) <= k` thì trả `s`.

In [12]:
from typing import List

class SolutionRecursive:
    def digitSum(self, s: str, k: int) -> str:
        if len(s) <= k:
            return s
        merged = "".join(str(sum(map(int, s[i:i + k]))) for i in range(0, len(s), k))
        return self.digitSum(merged, k)

print(SolutionRecursive().digitSum("11111222223", 3))
print(SolutionRecursive().digitSum("00000000", 3))

135
000


**Độ phức tạp:** Time giống Cách 2; Space `O(n + số_vòng)` do ngăn xếp đệ quy. Số vòng nhỏ nên không lo tràn ngăn xếp (`≤ 100` vòng).

### Cách 4 — Chia khối bằng `iter` và `islice`

Thay vì tính chỉ số, ta tạo một **iterator** trên chuỗi rồi liên tục lấy `k` phần tử bằng `islice(it, k)` cho đến khi iterator cạn. Kỹ thuật chia khối hay dùng khi dữ liệu là luồng.

In [13]:
from itertools import islice
from typing import List

class SolutionIslice:
    def digitSum(self, s: str, k: int) -> str:
        while len(s) > k:
            it = iter(s)
            parts = []
            while True:
                chunk = list(islice(it, k))
                if not chunk:
                    break
                parts.append(str(sum(map(int, chunk))))
            s = "".join(parts)
        return s

it = iter("11111222223")
print(list(islice(it, 3)), list(islice(it, 3)), list(islice(it, 3)), list(islice(it, 3)), list(islice(it, 3)))
print(SolutionIslice().digitSum("11111222223", 3))

['1', '1', '1'] ['1', '1', '2'] ['2', '2', '2'] ['2', '3'] []
135


**Độ phức tạp:** Time `O(n)` mỗi vòng, Space `O(n)`. `islice` lấy đúng `k` phần tử và để iterator "nhớ" vị trí; khối cuối ngắn hơn, khối rỗng báo hết dữ liệu.

### Cách 5 — Hàm `one_round` và vòng lặp "đến khi ổn định"

Tách một vòng thành hàm riêng `one_round(s, k)` (thuần túy, không đổi trạng thái), rồi vòng `while` gọi nó. Dễ **kiểm thử từng vòng riêng lẻ** và dễ gắn `print` để trace.

In [14]:
from typing import List

def one_round(s: str, k: int) -> str:
    return "".join(str(sum(map(int, s[i:i + k]))) for i in range(0, len(s), k))

class SolutionRound:
    def digitSum(self, s: str, k: int) -> str:
        while len(s) > k:
            s = one_round(s, k)
        return s

print(one_round("11111222223", 3), one_round("3465", 3))
print(SolutionRound().digitSum("11111222223", 3))

3465 135
135


**Độ phức tạp:** như Cách 2. Lợi ích chính là **cấu trúc code**: một hàm = một vòng đúng như phát biểu đề.

## 8. TRACE từng bước

Trace Cách 2 cho `s = "11111222223"`, `k = 3`:

| vòng | `s` trước | `i` | `group = s[i:i+3]` | `sum` | `parts` |
|---|---|---|---|---|---|
| 1 | `11111222223` | 0 | `111` | 3 | `[3]` |
| 1 | | 3 | `112` | 4 | `[3,4]` |
| 1 | | 6 | `222` | 6 | `[3,4,6]` |
| 1 | | 9 | `23` | 5 | `[3,4,6,5]` |
| 2 | `3465` | 0 | `346` | 13 | `[13]` |
| 2 | | 3 | `5` | 5 | `[13,5]` |

Sau vòng 2: `s = "135"`, `len = 3 ≤ 3` → trả về `"135"`.

In [15]:
def trace_slice(s, k):
    print(f"bắt đầu: s={s!r}, k={k}")
    rnd = 0
    while len(s) > k:
        rnd += 1
        print(f"-- vòng {rnd}: s={s!r} (dài {len(s)} > {k})")
        parts = []
        for i in range(0, len(s), k):
            group = s[i:i + k]
            total = sum(map(int, group))
            parts.append(str(total))
            print(f"   i={i}: group={group!r} sum={total} parts={parts}")
        s = "".join(parts)
        print(f"   s mới = {s!r}")
    print(f"dừng vì len(s)={len(s)} <= k={k}")
    return s

print("kết quả:", trace_slice("11111222223", 3))

bắt đầu: s='11111222223', k=3
-- vòng 1: s='11111222223' (dài 11 > 3)
   i=0: group='111' sum=3 parts=['3']
   i=3: group='112' sum=4 parts=['3', '4']
   i=6: group='222' sum=6 parts=['3', '4', '6']
   i=9: group='23' sum=5 parts=['3', '4', '6', '5']
   s mới = '3465'
-- vòng 2: s='3465' (dài 4 > 3)
   i=0: group='346' sum=13 parts=['13']
   i=3: group='5' sum=5 parts=['13', '5']
   s mới = '135'
dừng vì len(s)=3 <= k=3
kết quả: 135


In [16]:
print("kết quả:", trace_slice("9999", 2))
print("kết quả:", trace_slice("00000000", 3))

bắt đầu: s='9999', k=2
-- vòng 1: s='9999' (dài 4 > 2)
   i=0: group='99' sum=18 parts=['18']
   i=2: group='99' sum=18 parts=['18', '18']
   s mới = '1818'
-- vòng 2: s='1818' (dài 4 > 2)
   i=0: group='18' sum=9 parts=['9']
   i=2: group='18' sum=9 parts=['9', '9']
   s mới = '99'
dừng vì len(s)=2 <= k=2
kết quả: 99
bắt đầu: s='00000000', k=3
-- vòng 1: s='00000000' (dài 8 > 3)
   i=0: group='000' sum=0 parts=['0']
   i=3: group='000' sum=0 parts=['0', '0']
   i=6: group='00' sum=0 parts=['0', '0', '0']
   s mới = '000'
dừng vì len(s)=3 <= k=3
kết quả: 000


## 9. Code nộp LeetCode

In [17]:
from typing import List

class Solution:
    def digitSum(self, s: str, k: int) -> str:
        while len(s) > k:
            s = "".join(str(sum(map(int, s[i:i + k]))) for i in range(0, len(s), k))
        return s

print(Solution().digitSum("11111222223", 3))
print(Solution().digitSum("00000000", 3))

135
000


## 10. Vì sao thuật toán đúng

- **Mỗi vòng đúng đề:** `range(0, len(s), k)` cho các chỉ số đầu nhóm theo thứ tự, `s[i:i+k]` lấy đủ `k` ký tự (hoặc phần còn lại ở nhóm cuối), `sum(map(int, ...))` là tổng chữ số, và `join` ghép theo thứ tự. Đây chính là ba bước của đề.
- **Điều kiện dừng đúng:** đề yêu cầu lặp **khi `len(s) > k`**, thuật toán dùng đúng điều kiện đó, nên khi `len(s) == k` thì không làm thêm vòng nào (lỗi hay gặp: dùng `>=`).
- **Luôn dừng (có số hạng giảm):** gọi `T` là tổng tất cả chữ số của `s`. Một nhóm có tổng `g` được thay bằng các chữ số của `g`, mà tổng chữ số của `g` **không lớn hơn** `g` (bằng `g` khi `g < 10`, nhỏ hơn khi `g ≥ 10`). Vậy `T` không bao giờ tăng. Nếu `T` không đổi thì mọi tổng nhóm đều `< 10`, mỗi nhóm trở thành **đúng một ký tự**, nên độ dài mới là `ceil(len/k) < len` vì `k ≥ 2` và `len > k ≥ 2`. Mỗi vòng, hoặc `T` giảm thật sự, hoặc độ dài giảm thật sự, mà cả hai là số nguyên không âm nên quá trình phải kết thúc.
- **Không có trường hợp rỗng:** `len(s) ≥ 1` và mỗi nhóm cho ít nhất một chữ số (tổng `0` cũng là `"0"`), nên chuỗi mới không bao giờ rỗng.

## 11. Phân tích Time / Space Complexity

Đặt `n = len(s)` ban đầu (`≤ 100`), `R` = số vòng.

| Cách | Time | Space |
|---|---|---|
| 1. Tính tay từng ký tự | `O(n · R)` | `O(n)` |
| 2. Lát cắt + `map/sum/join` | `O(n · R)` | `O(n)` |
| 3. Đệ quy | `O(n · R)` | `O(n + R)` |
| 4. `iter` + `islice` | `O(n · R)` | `O(n)` |
| 5. Hàm `one_round` | `O(n · R)` | `O(n)` |

**Số vòng `R`:** mỗi vòng thay `k` chữ số bằng tối đa `⌈log10(9k)⌉ + 1` chữ số. Khi `k` lớn, chuỗi co rất nhanh (`R` nhỏ). Khi `k = 2`, nhóm `99 → 18` không co, nhưng tổng chữ số `T` giảm, nên `R` vẫn nhỏ (hằng số thực tế với `n ≤ 100`). Mỗi vòng độ dài không tăng đáng kể, nên tổng thời gian cỡ vài nghìn phép toán nhỏ — tức thời.

## 12. Bộ test

Ví dụ đề, edge case (`len(s) <= k` ngay từ đầu, `len(s)` đúng bằng bội của `k`, tổng nhiều chữ số), và test ngẫu nhiên đối chiếu **tất cả cách** với một mô phỏng độc lập.

In [18]:
import random

solvers = [SolutionManual(), SolutionSlice(), SolutionRecursive(), SolutionIslice(), SolutionRound(), Solution()]

def reference(s, k):
    while len(s) > k:
        nxt = []
        for i in range(0, len(s), k):
            nxt.append(str(sum(int(c) for c in s[i:i + k])))
        s = "".join(nxt)
    return s

fixed = [
    ("11111222223", 3, "135"),
    ("00000000", 3, "000"),
    ("1", 2, "1"),                    # đã đủ ngắn: không làm vòng nào
    ("12", 2, "12"),                  # len == k: dừng ngay, KHÔNG gộp
    ("123", 2, "33"),                 # nhóm "12","3" -> "3","3" => "33" (dài 2, dừng)
    ("9999", 2, "99"),                # "99","99" -> "1818" -> "9","9" -> "99"
    ("9" * 100, 100, "9" * 100),      # len == k == 100
]
for s, k, expected in fixed:
    assert reference(s, k) == expected, (s, k, reference(s, k), expected)
    for sol in solvers:
        assert sol.digitSum(s, k) == expected, (type(sol).__name__, s, k)
print("test cố định: OK")
for sol in solvers:
    assert sol.digitSum("9" * 100, 99) == reference("9" * 100, 99)
print("ví dụ lớn:", reference("9" * 100, 99))

random.seed(2243)
for _ in range(500):
    n = random.randint(1, 30)
    k = random.randint(2, 12)
    s = "".join(random.choice("0123456789") for _ in range(n))
    expected = reference(s, k)
    for sol in solvers:
        assert sol.digitSum(s, k) == expected, (type(sol).__name__, s, k)
for _ in range(200):
    n = random.randint(1, 100)
    k = random.randint(2, 100)
    s = "".join(random.choice("0123456789") for _ in range(n))
    vals = {sol.digitSum(s, k) for sol in solvers}
    assert len(vals) == 1 and len(vals.pop()) <= k, (s, k)
print("700 test ngẫu nhiên: OK")

test cố định: OK
ví dụ lớn: 8919
700 test ngẫu nhiên: OK


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Điều kiện dừng sai (`len(s) >= k`)

Với `len(s) == k` đề **đã dừng**; dùng `>=` làm thêm một vòng không cần thiết (và còn có thể lặp mãi khi `len(s) == k`).

In [19]:
def wrong1(s, k):
    rounds = 0
    while len(s) >= k and rounds < 5:         # rounds để tránh lặp vô hạn trong demo
        s = "".join(str(sum(map(int, s[i:i + k]))) for i in range(0, len(s), k))
        rounds += 1
    return s

print("sai :", wrong1("12", 2), "(đáng lẽ giữ nguyên '12')")
print("đúng:", Solution().digitSum("12", 2))

sai : 3 (đáng lẽ giữ nguyên '12')
đúng: 12


### Lỗi 2: Bỏ sót nhóm cuối

Nếu chỉ duyệt `range(0, len(s) - k + 1, k)` (chỉ các nhóm đủ `k`), nhóm cuối ngắn hơn bị mất.

In [20]:
def wrong2(s, k):
    while len(s) > k:
        s = "".join(str(sum(map(int, s[i:i + k]))) for i in range(0, len(s) - k + 1, k))
    return s

print("sai :", wrong2("11111222223", 3), "(mất nhóm '23')")
print("đúng:", Solution().digitSum("11111222223", 3))

sai : 346 (mất nhóm '23')
đúng: 135


### Lỗi 3: Cộng ký tự mà không đổi sang số

`sum(group)` với `group` là chuỗi gây `TypeError`; còn dùng `"1" + "1"` thì ra chuỗi `"11"`, không phải `2`.

In [21]:
group = "112"
try:
    print(sum(group))
except TypeError as e:
    print("sai -> TypeError:", e)
print("sai âm thầm:", "1" + "1")
print("đúng:", sum(map(int, group)), sum(int(c) for c in group))

sai -> TypeError: unsupported operand type(s) for +: 'int' and 'str'
sai âm thầm: 11
đúng: 4 4


### Lỗi 4: Quên `str(...)` khi ghép tổng

`"".join([3, 4, 6, 5])` lỗi vì phần tử là số nguyên.

In [22]:
try:
    print("".join([3, 4, 6, 5]))
except TypeError as e:
    print("sai -> TypeError:", e)
print("đúng:", "".join(map(str, [3, 4, 6, 5])), "".join(str(x) for x in [13, 5]))

sai -> TypeError: sequence item 0: expected str instance, int found
đúng: 3465 135


### Lỗi 5: Tổng có nhiều chữ số bị tưởng là một chữ số

Tổng `13` phải thêm **hai** ký tự `"1","3"` vào chuỗi mới. Vì vậy độ dài chuỗi mới có thể lớn hơn số nhóm; đừng giả sử mỗi nhóm cho một ký tự.

In [23]:
print(len("".join(str(x) for x in [13, 5])), "<- hai nhóm nhưng ba ký tự")
print(Solution().digitSum("9999", 2))

3 <- hai nhóm nhưng ba ký tự
99


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Đề dạng "**lặp quy trình cho đến khi ...**" → vòng `while` với điều kiện dừng đúng như đề; viết **hàm một vòng** rồi lặp.
- "Chia thành các nhóm liên tiếp độ dài `k`" → `for i in range(0, len(s), k): s[i:i+k]`. Lát cắt tự xử lý nhóm cuối ngắn.
- Chuỗi chữ số: `map(int, s)`, `sum(...)`, và `"".join(map(str, ...))` là bộ ba rất hay dùng.
- Đọc kỹ điều kiện dừng: `>` khác `>=`; thử ngay trường hợp `len(s) == k`.
- Muốn kiểm tra nhanh: viết hàm `reference` ngắn gọn rồi so với lời giải chính trên input ngẫu nhiên (đúng như phần test của notebook này).

## 15. Biến thể của bài

1. Hỏi **số vòng** cần thiết thay vì chuỗi cuối cùng.
2. Thay "tổng chữ số" bằng **tích** hoặc **tổng bình phương** (nhớ đến "Happy Number", LeetCode 202).
3. LeetCode 258 Add Digits: lặp tổng chữ số đến khi còn một chữ số (có công thức `1 + (n - 1) % 9`).
4. LeetCode 1945 Sum of Digits of String After Convert: đổi chữ cái thành số rồi cộng chữ số `k` lần.
5. LeetCode 2264/2255 (cùng đợt): các bài chuỗi cơ bản về nhóm ký tự và tiền tố.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Tính tay từng ký tự | cộng dồn với bộ đếm | `O(n·R)` | `O(n)` | hiểu bản chất, không dựa hàm dựng sẵn |
| 2. Lát cắt + `join` | `range(0, n, k)` + `s[i:i+k]` | `O(n·R)` | `O(n)` | **khuyến nghị khi nộp** |
| 3. Đệ quy | `digitSum` gọi chính nó | `O(n·R)` | `O(n + R)` | luyện tư duy đệ quy |
| 4. `iter` + `islice` | chia khối theo luồng | `O(n·R)` | `O(n)` | dữ liệu dạng luồng/iterator |
| 5. Hàm `one_round` | tách một vòng thành hàm | `O(n·R)` | `O(n)` | dễ test từng vòng, dễ trace |

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** `print([s[i:i+4] for i in range(0, 10, 4)])` với `s = "0123456789"` in ra gì?

**Bài 2 (tính tay).** `s = "8888"`, `k = 2`. Viết `s` sau mỗi vòng và kết quả cuối.

**Bài 3 (điền TODO).** Hoàn thành hàm thực hiện **đúng một vòng**.

In [24]:
def one_round_ex(s, k):
    # TODO: chia nhóm k, thay mỗi nhóm bằng tổng chữ số (dạng chuỗi), ghép lại
    pass

# chưa gọi hàm ở đây để notebook chạy không lỗi

**Bài 4 (tự viết lại).** Viết `count_rounds(s, k)` trả về **số vòng** cần làm cho đến khi `len(s) <= k`.

In [25]:
def count_rounds(s, k):
    # TODO: tự viết
    pass

**Bài 5 (mini challenge).** Viết `digit_root_chain(n)` mô phỏng LeetCode 258: lặp "tổng các chữ số của `n`" đến khi `n < 10`, trả về `n`. Rồi kiểm tra bằng công thức `1 + (n - 1) % 9` với mọi `n` từ `1` đến `1000`.

In [26]:
def digit_root_chain(n):
    # TODO: while n >= 10: n = tổng chữ số của n
    pass

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: các chỉ số `0, 4, 8`; lát cắt cuối ngắn hơn. Bài 2: `88` → `16`, nên `s` sau vòng 1 là `1616`.
- **Mức 2 (vừa):** Bài 3: dùng `"".join(str(sum(map(int, s[i:i+k]))) for i in range(0, len(s), k))`. Bài 4: tăng biến đếm trong `while len(s) > k`.
- **Mức 3 (sâu):** Bài 5: `n = sum(map(int, str(n)))` trong vòng `while n >= 10`. Công thức `1 + (n - 1) % 9` vì tổng chữ số đồng dư với `n` theo mod 9.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [27]:
# Bài 1 và 2
s = "0123456789"
assert [s[i:i + 4] for i in range(0, 10, 4)] == ["0123", "4567", "89"]
assert one_round("8888", 2) == "1616"
assert one_round("1616", 2) == "77"                     # 1+6=7, 1+6=7
assert Solution().digitSum("8888", 2) == "77"

# Bài 3
def one_round_ex(s, k):
    return "".join(str(sum(map(int, s[i:i + k]))) for i in range(0, len(s), k))

# Bài 4
def count_rounds(s, k):
    rounds = 0
    while len(s) > k:
        s = one_round_ex(s, k)
        rounds += 1
    return rounds

# Bài 5
def digit_root_chain(n):
    while n >= 10:
        n = sum(map(int, str(n)))
    return n

assert one_round_ex("11111222223", 3) == "3465"
assert one_round_ex("3465", 3) == "135"
assert count_rounds("11111222223", 3) == 2
assert count_rounds("12", 2) == 0
assert count_rounds("8888", 2) == 2
assert digit_root_chain(38) == 2 and digit_root_chain(9) == 9
assert all(digit_root_chain(n) == 1 + (n - 1) % 9 for n in range(1, 1001))
print("tất cả bài tập OK")

tất cả bài tập OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (mô phỏng: lặp một vòng khi còn dài hơn `k`)

```python
class Solution:
    def digitSum(self, s: str, k: int) -> str:
        while len(s) > k:
            s = "".join(str(sum(map(int, s[i:i + k]))) for i in range(0, len(s), k))
        return s
```

**Checklist**

- [ ] Điều kiện lặp là `len(s) > k` (không phải `>=`)?
- [ ] Chia nhóm bằng `range(0, len(s), k)` và `s[i:i+k]` (nhóm cuối ngắn vẫn được xử lý)?
- [ ] Đổi từng ký tự sang số bằng `int` trước khi `sum`, và đổi tổng về chuỗi bằng `str` trước khi `join`?
- [ ] Hiểu rằng tổng nhóm có thể có **nhiều chữ số**?
- [ ] Đã thử `len(s) <= k` ngay từ đầu (trả về nguyên `s`) và `len(s) == k`?
- [ ] Đã thử toàn `0`, toàn `9`, và `k = 2`?
- [ ] Signature đúng `digitSum(self, s: str, k: int) -> str` và có `from typing import List`?